# جمع مصادر رسمية عن ذوي الإعاقة في السعودية

**الهدف:** جمع نص نظيف من صفحات رسمية (وزارات، صناديق، هيئات) عن حقوق وخدمات ذوي الإعاقة،
كخطوة أولى نحو قاعدة معرفة لنظام RAG (استرجاع من مصدر موثّق عند كل سؤال) — لا لتدريب نموذج.

**لماذا Colab لا خادمك أو جهازك؟** لأن Colab عنده اتصال إنترنت كامل ومباشر، بينما بيئة
التطوير التي كتبت منها هذا الدفتر محجوبة عن أي موقع خارجي — هذا الدفتر لم يُختبر فعلياً من هناك
لهذا السبب بالذات؛ شغّله وراجع مخرجاته بعناية أكبر من المعتاد.

**أدب الجمع (مهم — لا تُزل هذي الفحوص):**
- يتحقق من `robots.txt` لكل موقع قبل أي طلب، ويتخطّى ما يمنعه صراحة.
- ينتظر ثوانٍ بين كل طلبين — لا يُحمّل خوادم حكومية بطلبات متلاحقة.
- يعرّف نفسه بـ User-Agent واضح بدل انتحال متصفح.
- يجلب فقط الروابط المذكورة صراحة أدناه — لا يزحف تلقائياً داخل المواقع.

**القائمة أدناه بداية لا قائمة نهائية.** جُمعت من نتائج بحث، لا فهرسة كاملة لكل موقع —
أضف أي صفحة رسمية أخرى تصادفها بنفسك (خصوصاً صفحات تفصيلية عميقة داخل كل موقع، فالبحث
يسطح عادة على الصفحات الرئيسية فقط).

In [ ]:
%%capture
!pip install -U trafilatura beautifulsoup4 requests

## قائمة المصادر

أضف/احذف روابط حسب الحاجة. كل رابط جديد على سطر مستقل داخل قائمته.

In [ ]:
SEED_URLS = [
    # هيئة رعاية الأشخاص ذوي الإعاقة (APD) — الجهة الأكثر تخصصاً في هذا الملف تحديداً،
    # أُنشئت بقرار مجلس الوزراء رقم 266 وتاريخ 1439/5/27هـ
    "https://www.apd.gov.sa/",
    "https://www.apd.gov.sa/services",
    "https://apd.gov.sa/en/service",

    # منصة أبشر / الحكومة الإلكترونية
    "https://my.gov.sa/ar/content/disabilities",

    # وزارة الموارد البشرية والتنمية الاجتماعية (HRSD)
    "https://www.hrsd.gov.sa/en/knowledge-centre/decisions-and-regulations/regulation-and-procedures/%D9%86%D8%B8%D8%A7%D9%85-%D8%AD%D9%82%D9%88%D9%82-%D8%A7%D9%84%D8%A3%D8%B4%D8%AE%D8%A7%D8%B5-%D8%B0%D9%88%D9%8A-%D8%A7%D9%84%D8%A5%D8%B9%D8%A7%D9%82%D8%A9",
    "https://www.hrsd.gov.sa/en/ministry-services/services/%D8%A7%D9%84%D8%A5%D8%B9%D8%A7%D9%86%D8%A9-%D8%A7%D9%84%D9%85%D8%A7%D9%84%D9%8A%D8%A9-%D9%84%D9%84%D8%A3%D8%B4%D8%AE%D8%A7%D8%B5-%D8%B0%D9%88%D9%8A-%D8%A7%D9%84%D8%A5%D8%B9%D8%A7%D9%82%D8%A9",
    "https://www.hrsd.gov.sa/en/care-about-you/empowering-people-special-needs",

    # وزارة التعليم — دمج ذوي الإعاقة في التعليم العام
    "https://www.moe.gov.sa/ar/education/generaleducation/Pages/PeopleWithSpecialNeeds.aspx",
    "https://moe.gov.sa/ar/aboutus/personsandlifecycle/Pages/disabilities.aspx",

    # هيئة حقوق الإنسان
    "https://hrc.gov.sa/website/hrc-in-ksa/4",

    # صندوق تنمية الموارد البشرية (هدف) — برنامج وصول لدعم النقل
    "https://www.hrdf.org.sa/products-and-services/programs/individuals/enable/wusool/",
    "https://www.hrdf.org.sa/products-and-services/programs/",

    # بنك التنمية الاجتماعية — تمويل ذوي الإعاقة (منتج كنف)
    "https://www.sdb.gov.sa/ar/%D8%AA%D9%88%D8%A7%D8%B5%D9%84-%D9%85%D8%B9%D9%86%D8%A7/%D8%A7%D9%84%D8%A7%D8%B3%D9%8A%D9%84%D8%A9-%D8%A7%D9%84%D8%B4%D8%A7%D9%8A%D8%B9%D8%A9/?product=36908",

    # ⚠️ مركز الملك سلمان لأبحاث الإعاقة: لم أوثّق نطاقه الرسمي بثقة كافية من نتائج
    # البحث المتاحة لي — أضف رابط موقعهم الرسمي هنا بنفسك، خصوصاً أنه راعي الهاكاثون
    # نفسه فمحتواه مباشر الصلة بمعايير التحكيم.
]

print(f"عدد الروابط الابتدائية: {len(SEED_URLS)}")

## أدوات الجلب المهذَّب

In [ ]:
import time, json, os
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser
import requests
import trafilatura
from bs4 import BeautifulSoup

USER_AGENT = "WesalInnovation-ResearchBot/1.0 (+contact: info@wesalinnovation.sa)"
DELAY_SECONDS = 3
OUTPUT_DIR = "disability_sources"
os.makedirs(OUTPUT_DIR, exist_ok=True)

_robots_cache = {}

def allowed_by_robots(url: str) -> bool:
    origin = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
    if origin not in _robots_cache:
        rp = RobotFileParser()
        rp.set_url(origin + "/robots.txt")
        try:
            rp.read()
        except Exception:
            # تعذّر قراءة robots.txt (قد لا يكون موجوداً) — نفترض السماح افتراضياً
            _robots_cache[origin] = None
            return True
        _robots_cache[origin] = rp
    rp = _robots_cache[origin]
    return True if rp is None else rp.can_fetch(USER_AGENT, url)

def clean_extract(url: str, html: str):
    text = trafilatura.extract(html, include_comments=False, include_tables=True)
    if not text or len(text.strip()) <= 50:
        # احتياط: trafilatura أحياناً لا يستخرج شيئاً من صفحات ذات بنية غير معتادة
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
    try:
        # عنوان الصفحة منفصل عن النص لأن trafilatura.extract() يُسقطه أحياناً من
        # جسم النص رغم أنه أوضح إشارة لمحتوى الصفحة — مفيد لاحقاً كمصدر/رابط يُعرض للمستخدم.
        meta = trafilatura.extract_metadata(html)
        title = meta.title if meta else None
    except Exception:
        title = None
    return title, text

def fetch_one(url: str) -> dict:
    if not allowed_by_robots(url):
        return {"url": url, "ok": False, "error": "ممنوع بموجب robots.txt"}
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        return {"url": url, "ok": False, "error": str(e)}
    title, text = clean_extract(url, resp.text)
    return {
        "url": url,
        "ok": True,
        "title": title,
        "fetched_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "length": len(text),
        "text": text,
    }

## التشغيل

In [ ]:
results = []
for i, url in enumerate(SEED_URLS, 1):
    print(f"[{i}/{len(SEED_URLS)}] {url}")
    r = fetch_one(url)
    results.append(r)
    if r["ok"]:
        fname = f"{i:03d}.json"
        with open(os.path.join(OUTPUT_DIR, fname), "w", encoding="utf-8") as f:
            json.dump(r, f, ensure_ascii=False, indent=2)
        print(f"   ✓ {r['length']} حرفاً → {fname}")
    else:
        print(f"   ✗ {r['error']}")
    time.sleep(DELAY_SECONDS)

ok_count = sum(1 for r in results if r["ok"])
print(f"\nتم: {ok_count}/{len(SEED_URLS)} صفحة")

## تنزيل النتائج

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
files.download(f"{OUTPUT_DIR}.zip")

## الخطوات التالية

1. **راجع كل ملف JSON يدوياً.** الاستخراج التلقائي مو مثالياً على كل بنية صفحة — بعض
   الملفات قد تحوي نصاً ناقصاً أو قوائم تنقّل لم تُزَل بالكامل. صحّح أو احذف ما يلزم.
2. **وسّع قائمة `SEED_URLS`** بصفحات فرعية أعمق كلما وجدتها — هذي القائمة بداية فقط.
3. **بناء نظام RAG فعلي** (تقطيع النصوص، توليد embeddings، تخزينها في قاعدة بيانات متجهية،
   وربطها باستدعاء `chat.php` بحيث يسترجع المقطع الصحيح قبل الرد) — خطوة منفصلة تالية،
   أخبرني إذا تحب نبدأ فيها بعد ما تجمع مادة كافية.